# 00 - Harness reconnaissance (CPU only, no GPU quota)

Set the accelerator to **None**. Nothing here needs a GPU, and every question it
answers would otherwise cost GPU-hours of black-box probing at one submission per day.

The trick that makes it free: we never *import* the harness. vLLM needs CUDA, so we
locate the installed packages and read their `.py` files as text.

Questions this run must settle, in priority order:

1. The default `max_turns`. If it is well below `max_tool_calls=100`, turns are the
   real ceiling and raising them in `eval_config.yaml` is the cheapest win available.
2. Whether `eval_config.yaml` can *raise* 60 min / 100 calls or only lower them.
3. Whether sub-agent tool calls consume the parent's budget (decides whether a
   multi-specialist topology is affordable at all).
4. Whether `run_command` wall time counts against the 60-minute budget.
5. Whether `skills/` is really loaded, and whether `hints_text` is injected.
6. Whether `seed` reaches vLLM.
7. Whether the `!include` loader rejects `..` outright or only escapes from the root.
   The shipped spec says outright, but the bundle that scored 0.15 uses `../prompts/`.
8. Whether declarative YAML supports a `LoopAgent` early exit.

**An unanswered question stays unanswered. It does not quietly become an assumption.**

In [ ]:
import os, subprocess, sys
from pathlib import Path

INPUT = Path('/kaggle/input')
print('attached inputs:')
for p in sorted(INPUT.iterdir()) if INPUT.is_dir() else []:
    print('  ', p.name)


def find_dir(*needles, must_contain='*.whl'):
    """Locate an attached dataset by name fragment rather than a hardcoded path.

    The official starter uses /kaggle/input/datasets/metric/... which only exists
    inside the scoring environment; in a normal notebook the same data mounts under
    its own dataset slug. Searching makes this notebook work in both.
    """
    cands = []
    for root in (INPUT,):
        if not root.is_dir():
            continue
        for p in root.rglob('*'):
            if not p.is_dir():
                continue
            name = p.name.lower()
            if any(n in name for n in needles) and (not must_contain or any(p.glob(must_contain))):
                cands.append(p)
    return sorted(cands, key=lambda q: len(str(q)))[0] if cands else None


WHEELHOUSE = find_dir('wheelhouse', must_contain='*.whl')
LAB = find_dir('swegemma-lab', 'swegemma_lab', must_contain='')
print('\nwheelhouse:', WHEELHOUSE)
print('lab code  :', LAB)
assert WHEELHOUSE, 'attach the dataset metric/gemma-4-developer-agent-wheelhouse'
assert LAB, 'attach the dataset ahmedsherif382/swegemma-lab'

In [ ]:
# Kaggle strips the PEP 440 '+cu128' local version from wheel filenames on upload,
# so restore it via symlinks before installing. --no-deps means pip never tries to
# resolve CUDA packages, which is what keeps this a CPU-only job.
tmp = Path('/tmp/wheelhouse')
tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    target = tmp / name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp.glob('*.whl'))
print(f'found {len(wheels)} wheels')
assert wheels, f'no .whl files under {WHEELHOUSE}'
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *wheels],
    capture_output=True, text=True,
)
print('pip rc', r.returncode)
print(r.stdout[-2000:] or '(no stdout)')
print(r.stderr[-2000:] or '(no stderr)')

In [ ]:
SCRIPT = LAB / 'scripts' / 'inspect_harness.py'
OUT = Path('/kaggle/working/harness_facts')
OUT.mkdir(parents=True, exist_ok=True)   # so the archive cell cannot fail on a bad run
print('script:', SCRIPT, SCRIPT.is_file())
rc = subprocess.call([sys.executable, str(SCRIPT), '--out', str(OUT), '--context', '4'])
print('\nexit code', rc)

## Read the answers, then transcribe them

`harness_facts/source/` now holds every harness `.py` file, so no later question needs
another Kaggle session. `answers.json` holds the matched excerpts per question and
`defaults.json` every budget-shaped constant.

Copy each settled answer into `README.md` under **Verified harness facts**, with the
`file:line` it came from. That section is the project's source of truth; the shipped
`HARNESS_README.md` is documentation, and we already suspect it is wrong about `..`
in `!include` paths.

In [ ]:
import re


def grep(pattern, context=4, limit=12):
    """Follow-up search over the saved sources, for anything the canned questions missed."""
    rx = re.compile(pattern)
    shown = 0
    for p in sorted((OUT / 'source').glob('*.py')):
        lines = p.read_text(encoding='utf-8', errors='replace').splitlines()
        for i, line in enumerate(lines):
            if rx.search(line):
                lo, hi = max(0, i - context), min(len(lines), i + context + 1)
                print(f'-- {p.name}:{i + 1}')
                for n in range(lo, hi):
                    print(f'{n + 1:5d}| {lines[n]}')
                print()
                shown += 1
                if shown >= limit:
                    return
    if not shown:
        print('no matches')


grep(r'def load_submission_eval_config|def build_submission_limits')

In [ ]:
# Bundle the findings so the next session starts from them rather than repeating this.
import shutil

n = len(list(OUT.rglob('*')))
print(f'{n} files under {OUT}')
if n:
    shutil.make_archive('/kaggle/working/harness_facts', 'zip', root_dir=OUT)
    print('wrote /kaggle/working/harness_facts.zip')
    print('pull it locally with:')
    print('  kaggle kernels output ahmedsherif382/swegemma-harness-recon -p reference/harness_facts')
else:
    print('nothing to archive -- the inspection step did not produce output')